# Person 4: PCA and K-Means clustering (development partition)

**Owner:** Nathan Pak · DATA 245 Kepler exoplanet classification

This notebook covers the Person 4 workstream from the proposal:

1. Standardize the 76 leakage-aware features and run PCA
2. Explained variance and the number of components for 90% and 95% of variance
3. 2D PCA visualization coloured by disposition, and the top loadings
4. **Original vs PCA-reduced features** (novelty experiment) under the saved grouped CV folds
5. K-Means clustering without labels, then comparison with the true dispositions

**Rules followed (from `docs/shared/TEAM_WORKFLOW.md`):** development partition only (7,381 rows with a measured transit depth), the test partition is never loaded or scored, seed 245, and for every supervised score the imputer, scaler and PCA are fit inside each training fold. Figures go to `figures/person4/`, metrics to `results/person4/`.

All results are **development-stage** and not final team results.

In [1]:
import sys, json, warnings
from pathlib import Path

# Locate the repository root so `src/` is importable from this notebook.
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "exoplanet").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Run this notebook from inside the kepler-exoplanet-classification repository.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score,
                             silhouette_score)

from exoplanet.config import CLASS_ORDER, RANDOM_SEED, TARGET_COLUMN
from exoplanet.development import load_development_bundle, development_fold_masks
from exoplanet.evaluation import cross_validate_estimator, write_experiment_results
from exoplanet.models.person4_pca_clustering.transforms import make_pca_preprocess

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = RANDOM_SEED  # 245
FIG = ROOT / "figures" / "person4"; FIG.mkdir(parents=True, exist_ok=True)
RES = ROOT / "results" / "person4"; RES.mkdir(parents=True, exist_ok=True)

# Chart styling: fixed colour per class (never reassigned), recessive grid.
CLASS_COLOR = {"CONFIRMED": "#2a78d6", "CANDIDATE": "#eb6834", "FALSE POSITIVE": "#1baf7a"}
CLASS_MARKER = {"CONFIRMED": "o", "CANDIDATE": "^", "FALSE POSITIVE": "s"}
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#8a8984", "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.6,
    "axes.titlesize": 12, "axes.titleweight": "bold", "font.size": 10,
    "legend.frameon": False,
})
print("repo root:", ROOT)

repo root: /Users/nathanpak/Desktop/Data245/project/kepler-exoplanet-classification


## 1. Load the shared development bundle

In [2]:
bundle = load_development_bundle()
frame = bundle.frame
features = bundle.feature_columns
X = frame[features]
y = frame[TARGET_COLUMN]

print(f"rows={bundle.n_rows}, host stars={bundle.n_groups}, features={len(features)}, "
      f"feature list status={bundle.proposal_status}")
print(y.value_counts().reindex(CLASS_ORDER))

rows=7381, host stars=6357, features=76, feature list status=proposed
koi_disposition
CONFIRMED         2197
CANDIDATE         1505
FALSE POSITIVE    3679
Name: count, dtype: int64


## 2. Preprocessing choice: plain standardization vs signed-log + standardization

Both variants use the shared median imputer and `StandardScaler`. The second first applies `sign(x)·log1p(|x|)` to columns whose |skewness| > 3 (learned from the rows it is fit on). The reason is that many KOI columns (period, depth, insolation, centroid offsets and their errors) are heavy-tailed, and a handful of extreme rows can dominate both PCA and K-Means.

For the **exploratory** sections (PCA variance, 2D plots, K-Means) the pipeline is fit on all development rows. That is unsupervised and uses no labels and no test rows. For the **supervised comparison** in section 5, everything is refit inside each training fold.

In [3]:
variants = {
    "standardized": make_pca_preprocess(log_skewed=False),
    "signed-log + standardized": make_pca_preprocess(log_skewed=True),
}
Z = {name: pipe.fit_transform(X) for name, pipe in variants.items()}
log_cols = variants["signed-log + standardized"].named_steps["signed_log"].columns_
print(f"signed-log applied to {len(log_cols)} of {len(features)} columns")

skew = X.skew().abs().sort_values(ascending=False)
print("most skewed columns:"); print(skew.head(8).round(1).to_string())

signed-log applied to 53 of 76 columns
most skewed columns:
koi_period        85.1
koi_dor           82.1
koi_kmag_err      82.0
koi_dor_err1      81.6
koi_insol_err2    59.1
koi_prad_err2     56.0
koi_prad          53.7
koi_sma           53.6


## 3. PCA: explained variance

In [4]:
pcas, var_rows = {}, []
for name, z in Z.items():
    p = PCA(random_state=SEED).fit(z)
    pcas[name] = p
    cum = np.cumsum(p.explained_variance_ratio_)
    var_rows.append({
        "preprocessing": name,
        "PC1": p.explained_variance_ratio_[0], "PC2": p.explained_variance_ratio_[1],
        "PC1+PC2": cum[1], "first 10 PCs": cum[9],
        "components for 80%": int(np.argmax(cum >= 0.80) + 1),
        "components for 90%": int(np.argmax(cum >= 0.90) + 1),
        "components for 95%": int(np.argmax(cum >= 0.95) + 1),
    })
variance_table = pd.DataFrame(var_rows).set_index("preprocessing")
variance_table.to_csv(RES / "pca_explained_variance_summary.csv")
pd.DataFrame({name: p.explained_variance_ratio_ for name, p in pcas.items()},
             index=pd.RangeIndex(1, len(features) + 1, name="component")
             ).to_csv(RES / "pca_explained_variance_by_component.csv")
variance_table.round(3)

,PC1,PC2,PC1+PC2,first 10 PCs,components for 80%,components for 90%,components for 95%
preprocessing,,,,,,,
standardized,0.128,0.087,0.216,0.582,21,31,38
signed-log + standardized,0.164,0.134,0.298,0.691,16,25,33


In [5]:
main = "signed-log + standardized"
p = pcas[main]
cum = np.cumsum(p.explained_variance_ratio_)
k = np.arange(1, len(cum) + 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].bar(k, p.explained_variance_ratio_ * 100, color="#2a78d6", width=0.8)
axes[0].set(title="Variance explained per component", xlabel="Principal component",
            ylabel="% of variance", xlim=(0, 40))

for name, style in [("signed-log + standardized", "-"), ("standardized", "--")]:
    c = np.cumsum(pcas[name].explained_variance_ratio_) * 100
    axes[1].plot(k, c, style, color="#2a78d6" if style == "-" else "#8a8984", lw=2, label=name)
for thr in (90, 95):
    n = int(np.argmax(cum * 100 >= thr) + 1)
    axes[1].axhline(thr, color="#c3c2b7", lw=0.8)
    axes[1].annotate(f"{thr}% at {n} PCs", (n, thr), xytext=(n + 3, thr - 9),
                     color="#52514e", fontsize=9, arrowprops=dict(arrowstyle="-", color="#8a8984"))
axes[1].set(title="Cumulative explained variance", xlabel="Number of components",
            ylabel="Cumulative %", ylim=(0, 101))
axes[1].legend(loc="lower right")
fig.tight_layout(); fig.savefig(FIG / "pca_explained_variance.png"); plt.show()

/var/folders/2z/4zwc3r9j73ng08qw2l8py_0h0000gn/T/ipykernel_56721/2388340070.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIG / "pca_explained_variance.png"); plt.show()


**Reading:** variance is spread across many directions. No single component dominates, which is expected because the 76 features mix stellar, transit, photometric and centroid-offset measurements that are only partly correlated. The proposal's 90–95% target needs roughly a third to half of the original dimensions.

## 4. 2D projection and what the first components mean

In [6]:
scores = p.transform(Z[main])[:, :2]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.4), sharex=True, sharey=True)
rng = np.random.default_rng(SEED)
for ax, cls in zip(axes, CLASS_ORDER):
    # grey context of all rows, then the highlighted class on top (small multiples
    # avoid three overlapping colour clouds)
    ax.scatter(scores[:, 0], scores[:, 1], s=4, color="#dcdbd6", lw=0, rasterized=True)
    m = (y == cls).to_numpy()
    ax.scatter(scores[m, 0], scores[m, 1], s=6, color=CLASS_COLOR[cls], alpha=0.55,
               lw=0, marker=CLASS_MARKER[cls], rasterized=True)
    ax.set_title(f"{cls} (n={m.sum():,})", color="#0b0b0b")
    ax.set_xlabel(f"PC1 ({p.explained_variance_ratio_[0]:.1%})")
axes[0].set_ylabel(f"PC2 ({p.explained_variance_ratio_[1]:.1%})")
lo, hi = np.percentile(scores, [0.5, 99.5], axis=0)
axes[0].set_xlim(lo[0], hi[0]); axes[0].set_ylim(lo[1], hi[1])
fig.suptitle("Development KOIs on the first two principal components (signed-log + standardized)",
             x=0.01, ha="left", fontweight="bold")
fig.tight_layout(); fig.savefig(FIG / "pca_2d_by_disposition.png"); plt.show()

centroids = pd.DataFrame(scores, columns=["PC1", "PC2"]).groupby(y.to_numpy()).mean().reindex(CLASS_ORDER)
centroids.round(2)

/var/folders/2z/4zwc3r9j73ng08qw2l8py_0h0000gn/T/ipykernel_56721/854184301.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIG / "pca_2d_by_disposition.png"); plt.show()


,PC1,PC2
CONFIRMED,0.78,-0.60
CANDIDATE,1.74,1.81
FALSE POSITIVE,-1.17,-0.38


In [7]:
loadings = pd.DataFrame(p.components_[:3].T, index=features, columns=["PC1", "PC2", "PC3"])
loadings.to_csv(RES / "pca_loadings_pc1_pc3.csv")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, pc in zip(axes, ["PC1", "PC2"]):
    top = loadings[pc].reindex(loadings[pc].abs().sort_values(ascending=False).index[:10])[::-1]
    ax.barh(top.index, top.values, color=["#2a78d6" if v > 0 else "#eb6834" for v in top.values], height=0.7)
    ax.axvline(0, color="#8a8984", lw=0.8)
    ax.set(title=f"Top 10 loadings on {pc}", xlabel="Loading")
    ax.grid(axis="y", visible=False)
fig.tight_layout(); fig.savefig(FIG / "pca_top_loadings.png"); plt.show()

for pc in ["PC1", "PC2", "PC3"]:
    top = loadings[pc].abs().sort_values(ascending=False).index[:6]
    print(pc, ":", ", ".join(f"{c} ({loadings.loc[c, pc]:+.2f})" for c in top))

PC1 : koi_insol_err2 (+0.22), koi_teq (-0.21), koi_insol (-0.21), koi_insol_err1 (-0.21), koi_prad_err2 (+0.19), koi_srad_err2 (+0.19)
PC2 : koi_dikco_msky_err (+0.21), koi_dikco_mdec_err (+0.20), koi_dicco_msky_err (+0.20), koi_kepmag (-0.20), koi_rmag (-0.20), koi_dikco_mra_err (+0.19)
PC3 : koi_period (+0.25), koi_depth_err1 (+0.25), koi_sma (+0.25), koi_prad (+0.24), koi_prad_err1 (+0.23), koi_dor (+0.23)


/var/folders/2z/4zwc3r9j73ng08qw2l8py_0h0000gn/T/ipykernel_56721/4018713270.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIG / "pca_top_loadings.png"); plt.show()


## 5. Novelty experiment: original features vs PCA-reduced features

Same probe classifier, same saved grouped folds, only the input changes. The probe is a multinomial Logistic Regression (`class_weight="balanced"`, `max_iter=3000`). It is used here only to measure how much class information PCA keeps. It is **not** a replacement for Person 1's Logistic Regression results.

Inside each training fold, the pipeline fits the signed-log step, the median imputer, the scaler and PCA, then the classifier. Validation folds never influence any of those steps. The primary metric is grouped validation macro-F1, as agreed in the README.

In [8]:
k90 = int(variance_table.loc[main, "components for 90%"])
k95 = int(variance_table.loc[main, "components for 95%"])
component_grid = sorted({2, 3, 5, 10, 15, 20, k90, k95})

def probe_factory(n_components=None, log_skewed=True):
    def factory():
        steps = list(make_pca_preprocess(log_skewed=log_skewed).steps)
        if n_components is not None:
            steps.append(("pca", PCA(n_components=n_components, random_state=SEED)))
        steps.append(("model", LogisticRegression(max_iter=3000, class_weight="balanced")))
        return Pipeline(steps)
    return factory

runs = [("original 76 features", None)] + [(f"PCA {n} components", n) for n in component_grid]
rows = []
for label, n in runs:
    s = cross_validate_estimator(bundle, probe_factory(n))
    name = "original_76" if n is None else f"pca_{n:02d}"
    write_experiment_results("person4", f"pca_vs_original/{name}", s, {
        "model": "LogisticRegression(max_iter=3000, class_weight='balanced')",
        "preprocessing": "signed-log(|skew|>3) + median impute + StandardScaler, fit per training fold",
        "pca_components": n, "random_state": SEED,
        "purpose": "probe of information retained by PCA; not the Person 1 baseline"})
    per_class = {r["class"]: r["recall"] for r in s["per_class"]}
    rows.append({"features": label, "n_inputs": 76 if n is None else n,
                 "val_macro_f1": s["mean_validation_macro_f1"], "val_macro_f1_sd": s["std_validation_macro_f1"],
                 "val_accuracy": s["mean_validation_accuracy"], "train_macro_f1": s["mean_train_macro_f1"],
                 **{f"recall_{c}": per_class[c] for c in CLASS_ORDER}})
comparison = pd.DataFrame(rows)
comparison.to_csv(RES / "pca_vs_original_comparison.csv", index=False)
comparison.round(4)

,features,n_inputs,val_macro_f1,val_macro_f1_sd,val_accuracy,train_macro_f1,recall_CONFIRMED,recall_CANDIDATE,recall_FALSE POSITIVE
0,original 76 features,76,0.7735,0.0108,0.7950,0.7879,0.8803,0.7136,0.7774
1,PCA 2 components,2,0.5704,0.0195,0.5784,0.5693,0.5699,0.6764,0.5434
2,PCA 3 components,3,0.5794,0.0108,0.5908,0.5787,0.6846,0.5980,0.5319
3,PCA 5 components,5,0.6629,0.0138,0.6772,0.6649,0.7833,0.6485,0.6254
4,PCA 10 components,10,0.6831,0.0138,0.6990,0.6859,0.7920,0.6678,0.6562
5,PCA 15 components,15,0.7537,0.0179,0.7740,0.7535,0.8639,0.7063,0.7480
6,PCA 20 components,20,0.7619,0.0128,0.7820,0.7633,0.8762,0.7163,0.7527
7,PCA 25 components,25,0.7664,0.0129,0.7874,0.7694,0.8858,0.7090,0.7608
8,PCA 33 components,33,0.7661,0.0128,0.7874,0.7708,0.8853,0.7063,0.7622


In [9]:
pc_rows = comparison[comparison.features.str.startswith("PCA")]
base = comparison.loc[comparison.features.eq("original 76 features")].iloc[0]

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.errorbar(pc_rows.n_inputs, pc_rows.val_macro_f1, yerr=pc_rows.val_macro_f1_sd, color="#2a78d6",
            lw=2, marker="o", ms=7, capsize=3, label="PCA-reduced inputs")
ax.axhline(base.val_macro_f1, color="#eb6834", lw=2, ls="--",
           label=f"original 76 features ({base.val_macro_f1:.3f})")
for n, lab in [(k90, "90%"), (k95, "95%")]:
    v = pc_rows.loc[pc_rows.n_inputs.eq(n), "val_macro_f1"].iloc[0]
    ax.annotate(f"{lab} var.\n{n} PCs: {v:.3f}", (n, v), xytext=(0, -38), textcoords="offset points",
                ha="center", fontsize=9, color="#52514e")
ax.set(title="Grouped validation macro-F1: PCA inputs vs original features",
       xlabel="Number of principal components", ylabel="Macro-F1 (mean ± sd, 5 folds)")
ax.legend(loc="lower right")
fig.tight_layout(); fig.savefig(FIG / "pca_vs_original_macro_f1.png"); plt.show()

/var/folders/2z/4zwc3r9j73ng08qw2l8py_0h0000gn/T/ipykernel_56721/3897915615.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIG / "pca_vs_original_macro_f1.png"); plt.show()


## 6. K-Means: choosing k without labels

K-Means is fit on the standardized features with no labels. For each k we record inertia (elbow), silhouette (on a 3,000-row sample for speed), and the size of the smallest cluster. Only afterwards are the clusters compared with the dispositions (ARI and NMI: 0 means no agreement beyond chance, 1 means identical partitions).

In [10]:
k_rows = []
for name, z in Z.items():
    for kk in range(2, 11):
        km = KMeans(n_clusters=kk, n_init=10, random_state=SEED).fit(z)
        sizes = np.bincount(km.labels_)
        k_rows.append({"preprocessing": name, "k": kk, "inertia": km.inertia_,
                       "silhouette": silhouette_score(z, km.labels_, sample_size=3000, random_state=SEED),
                       "smallest_cluster": int(sizes.min()),
                       "ARI_vs_labels": adjusted_rand_score(y, km.labels_),
                       "NMI_vs_labels": normalized_mutual_info_score(y, km.labels_)})
k_table = pd.DataFrame(k_rows)
k_table.to_csv(RES / "kmeans_k_selection.csv", index=False)
k_table.pivot(index="k", columns="preprocessing",
              values=["silhouette", "smallest_cluster", "ARI_vs_labels"]).round(3)

silhouette               \
preprocessing signed-log + standardized standardized   
k                                                      
2                                 0.148        0.166   
3                                 0.152        0.161   
4                                 0.138        0.161   
5                                 0.090        0.158   
6                                 0.101        0.072   
7                                 0.087        0.074   
8                                 0.087        0.084   
9                                 0.093        0.049   
10                                0.095        0.084   

                       smallest_cluster               \
preprocessing signed-log + standardized standardized   
k                                                      
2                                2224.0       2503.0   
3                                1145.0         26.0   
4                                1050.0          1.0   
5                                 902.0          1.0   
6                                 216.0          1.0   
7                                 214.0          1.0   
8                                  98.0          1.0   
9                                  98.0          1.0   
10                                 75.0          1.0   

                          ARI_vs_labels               
preprocessing signed-log + standardized standardized  
k                                                     
2                                -0.006       -0.005  
3                                 0.036       -0.005  
4                                 0.117       -0.005  
5                                 0.172        0.016  
6                                 0.157        0.143  
7                                 0.129        0.136  
8                                 0.119        0.128  
9                                 0.117        0.113  
10                                0.110        0.123

In [11]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for name, style, col in [("signed-log + standardized", "-", "#2a78d6"), ("standardized", "--", "#8a8984")]:
    t = k_table[k_table.preprocessing.eq(name)]
    axes[0].plot(t.k, t.inertia / 1e3, style, color=col, lw=2, marker="o", ms=6, label=name)
    axes[1].plot(t.k, t.silhouette, style, color=col, lw=2, marker="o", ms=6, label=name)
    axes[2].plot(t.k, t.ARI_vs_labels, style, color=col, lw=2, marker="o", ms=6, label=name)
axes[0].set(title="Elbow: inertia", xlabel="k", ylabel="Inertia (thousands)")
axes[1].set(title="Silhouette (higher = tighter)", xlabel="k", ylabel="Silhouette")
axes[2].set(title="Agreement with dispositions", xlabel="k", ylabel="Adjusted Rand index")
axes[2].legend(loc="upper right", fontsize=8)
fig.tight_layout(); fig.savefig(FIG / "kmeans_k_selection.png"); plt.show()

/var/folders/2z/4zwc3r9j73ng08qw2l8py_0h0000gn/T/ipykernel_56721/3723171226.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIG / "kmeans_k_selection.png"); plt.show()


**Why the signed-log variant:** with plain standardization, K-Means spends clusters on a handful of extreme rows (see `smallest_cluster`, which drops to 1 or a few dozen KOIs). That says more about outliers than about structure. After the signed-log step the clusters are balanced, and agreement with the dispositions is higher.

## 7. K-Means with k = 3 compared with the three dispositions

In [12]:
K = 3
km3 = KMeans(n_clusters=K, n_init=20, random_state=SEED).fit(Z[main])
clusters = pd.Series(km3.labels_, index=frame.index, name="cluster")

ct = pd.crosstab(clusters, y).reindex(columns=CLASS_ORDER)
row_pct = ct.div(ct.sum(axis=1), axis=0)
ct.to_csv(RES / "kmeans_k3_crosstab.csv")
metrics_k3 = {
    "k": K, "preprocessing": main, "n_init": 20, "random_state": SEED,
    "ARI": adjusted_rand_score(y, clusters), "NMI": normalized_mutual_info_score(y, clusters),
    "silhouette_sample3000": silhouette_score(Z[main], clusters, sample_size=3000, random_state=SEED),
    "purity": float(ct.max(axis=1).sum() / ct.values.sum()),
    "majority_class_baseline": float((y == "FALSE POSITIVE").mean()),
    "cluster_sizes": ct.sum(axis=1).tolist(), "test_scored": False,
}
(RES / "kmeans_k3_metrics.json").write_text(json.dumps(metrics_k3, indent=2))
print(json.dumps({k: (round(v, 4) if isinstance(v, float) else v) for k, v in metrics_k3.items()}, indent=1))

fig, ax = plt.subplots(figsize=(6.2, 3.2))
im = ax.imshow(row_pct.values, cmap="Blues", vmin=0, vmax=1, aspect="auto")
for i in range(K):
    for j in range(3):
        v = row_pct.values[i, j]
        ax.text(j, i, f"{v:.0%}\n({ct.values[i, j]:,})", ha="center", va="center", fontsize=9,
                color="white" if v > 0.55 else "#0b0b0b")
ax.set_xticks(range(3), CLASS_ORDER); ax.set_yticks(range(K), [f"Cluster {i} (n={n:,})" for i, n in enumerate(ct.sum(axis=1))])
ax.grid(False); ax.set_title("Disposition mix inside each K-Means cluster (row %)")
fig.colorbar(im, ax=ax, fraction=0.04, format=lambda v, _: f"{v:.0%}")
fig.tight_layout(); fig.savefig(FIG / "kmeans_k3_vs_disposition.png"); plt.show()

{
 "k": 3,
 "preprocessing": "signed-log + standardized",
 "n_init": 20,
 "random_state": 245,
 "ARI": 0.0363,
 "NMI": 0.0876,
 "silhouette_sample3000": 0.1552,
 "purity": 0.4984,
 "majority_class_baseline": 0.4984,
 "cluster_sizes": [
  1147,
  1644,
  4590
 ],
 "test_scored": false
}


/var/folders/2z/4zwc3r9j73ng08qw2l8py_0h0000gn/T/ipykernel_56721/3240469863.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIG / "kmeans_k3_vs_disposition.png"); plt.show()


In [13]:
cluster_colors = ["#6250d6", "#eda100", "#e87ba4", "#008300", "#e34948"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharex=True, sharey=True)
for c in range(K):
    m = (clusters == c).to_numpy()
    axes[0].scatter(scores[m, 0], scores[m, 1], s=5, alpha=0.5, lw=0, color=cluster_colors[c],
                    label=f"Cluster {c}", rasterized=True)
for cls in CLASS_ORDER:
    m = (y == cls).to_numpy()
    axes[1].scatter(scores[m, 0], scores[m, 1], s=5, alpha=0.45, lw=0, color=CLASS_COLOR[cls],
                    marker=CLASS_MARKER[cls], label=cls, rasterized=True)
axes[0].set_title("K-Means clusters (k = 3, no labels used)"); axes[1].set_title("True dispositions")
for ax in axes:
    ax.set_xlabel("PC1"); ax.legend(markerscale=3, loc="upper right", fontsize=8)
axes[0].set_ylabel("PC2"); axes[0].set_xlim(lo[0], hi[0]); axes[0].set_ylim(lo[1], hi[1])
fig.tight_layout(); fig.savefig(FIG / "kmeans_k3_pca_space.png"); plt.show()

/var/folders/2z/4zwc3r9j73ng08qw2l8py_0h0000gn/T/ipykernel_56721/711684669.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIG / "kmeans_k3_pca_space.png"); plt.show()


### What distinguishes the clusters (medians of interpretable features, original units)

In [14]:
profile_cols = ["koi_period", "koi_depth", "koi_prad", "koi_duration", "koi_model_snr",
                "koi_teq", "koi_impact", "koi_steff", "koi_srad", "koi_dikco_msky", "koi_count"]
profile = X[profile_cols].groupby(clusters).median()
profile.insert(0, "n", clusters.value_counts().sort_index())
profile.to_csv(RES / "kmeans_k3_cluster_profiles.csv")
profile.round(2)

,n,koi_period,koi_depth,koi_prad,koi_duration,koi_model_snr,koi_teq,koi_impact,koi_steff,koi_srad,koi_dikco_msky,koi_count
cluster,,,,,,,,,,,,
0,1147,279.04,381.1,2.11,6.75,9.40,314.0,0.31,5838.0,1.00,1.40,1.0
1,1644,4.44,375.4,14.86,4.30,44.45,1555.5,0.71,6133.5,1.83,0.54,1.0
2,4590,7.34,451.1,2.17,3.41,26.80,877.5,0.53,5652.0,0.91,0.44,1.0


## 8. Stability check across the saved development folds

Refit K-Means (k = 3) on each training fold and assign the held-out validation fold to the nearest centroid. If the clustering is real structure, these assignments should agree with the full-data clustering (ARI close to 1) and the label agreement should be similar on every fold.

In [15]:
stab = []
for fold in range(5):
    tr, va = development_fold_masks(frame, fold)
    pipe = make_pca_preprocess(log_skewed=True).fit(X.loc[tr])
    km = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit(pipe.transform(X.loc[tr]))
    pred = km.predict(pipe.transform(X.loc[va]))
    stab.append({"fold": fold, "n_validation": int(va.sum()),
                 "ARI_vs_full_fit": adjusted_rand_score(clusters[va], pred),
                 "ARI_vs_labels": adjusted_rand_score(y[va], pred),
                 "NMI_vs_labels": normalized_mutual_info_score(y[va], pred)})
stability = pd.DataFrame(stab)
stability.to_csv(RES / "kmeans_k3_fold_stability.csv", index=False)
print(stability.drop(columns="fold").agg(["mean", "std"]).round(3))
stability.round(3)

      n_validation  ARI_vs_full_fit  ARI_vs_labels  NMI_vs_labels
mean      1476.200            0.940          0.035          0.089
std          9.418            0.043          0.010          0.022


,fold,n_validation,ARI_vs_full_fit,ARI_vs_labels,NMI_vs_labels
0,0,1486,0.950,0.043,0.115
1,1,1481,0.908,0.040,0.101
2,2,1481,0.886,0.018,0.056
3,3,1470,0.959,0.038,0.081
4,4,1463,0.996,0.038,0.094


## 9. Summary for the intermediate status report

In [16]:
best_pca = comparison[comparison.features.str.startswith("PCA")].sort_values("val_macro_f1").iloc[-1]
summary = {
    "pca_components_for_90pct": k90, "pca_components_for_95pct": k95,
    "pc1_pc2_variance": float(variance_table.loc[main, "PC1+PC2"]),
    "original_76_macro_f1": float(base.val_macro_f1),
    f"pca_{k90}_macro_f1": float(comparison.loc[comparison.n_inputs.eq(k90) & comparison.features.str.startswith("PCA"), "val_macro_f1"].iloc[0]),
    f"pca_{k95}_macro_f1": float(comparison.loc[comparison.n_inputs.eq(k95) & comparison.features.str.startswith("PCA"), "val_macro_f1"].iloc[0]),
    "pca_2_macro_f1": float(comparison.loc[comparison.n_inputs.eq(2), "val_macro_f1"].iloc[0]),
    "best_pca": best_pca.features, "best_pca_macro_f1": float(best_pca.val_macro_f1),
    "kmeans_k3_ARI": metrics_k3["ARI"], "kmeans_k3_NMI": metrics_k3["NMI"], "kmeans_k3_purity": metrics_k3["purity"],
    "kmeans_fold_stability_ARI_mean": float(stability.ARI_vs_full_fit.mean()),
    "test_scored": False,
}
(RES / "person4_summary.json").write_text(json.dumps(summary, indent=2))
pd.Series(summary).to_frame("value")

,value
pca_components_for_90pct,25
pca_components_for_95pct,33
pc1_pc2_variance,0.297921
original_76_macro_f1,0.773469
pca_25_macro_f1,0.766405
pca_33_macro_f1,0.766129
pca_2_macro_f1,0.570394
best_pca,PCA 25 components
best_pca_macro_f1,0.766405
kmeans_k3_ARI,0.036269
